# Phase 1: Spurgeon Continued Pretraining — Step 7: Training (Notebook B)

This notebook executes PEFT QLoRA continued pretraining on `unsloth/Qwen2.5-3B` using the Hugging Face Dataset compiled in Notebook A. It handles 4-bit quantization, sequence packing (for fast compute), and cross-session checkpoint resumption logic.

## 1. Install Dependencies

In [ ]:
# Install Unsloth and specific patched versions for Kaggle environment
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Model & PEFT Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
LORA_RANK = 32

# Load base model in 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "unsloth/Qwen2.5-3B",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype        = None,
    load_in_4bit = True,
)

# Apply the PEFT LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha   = 64,
    lora_dropout = 0, # Critical: set to 0 to enable fast custom Triton kernels
    bias         = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 3. Configure Dataset and Training Arguments

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_from_disk
import os
import shutil

# Kaggle mounts input datasets as read-only. SFTTrainer's internal dataset.map()
# attempts to write temporary cached files in the dataset folder, causing an OSError.
# To prevent this, we copy the dataset from /kaggle/input/ to the writable /kaggle/working/ first.
src_dataset_path = "/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_dataset"
local_dataset_path = "/kaggle/working/spurgeon_dataset"

if not os.path.exists(local_dataset_path):
    print(f"Copying dataset from {src_dataset_path} to writable path {local_dataset_path}...")
    shutil.copytree(src_dataset_path, local_dataset_path)
else:
    print(f"Dataset already exists at writable path: {local_dataset_path}")

# Load the dataset from the writable local directory
dataset = load_from_disk(local_dataset_path)

output_dir = "/kaggle/working/checkpoints"


## 4. Launch Training

In [ ]:
if PREV_RUN_CHECKPOINT:
    if os.path.exists(PREV_RUN_CHECKPOINT):
        print(f"Resuming training from checkpoint: {PREV_RUN_CHECKPOINT}")
        trainer.train(resume_from_checkpoint=PREV_RUN_CHECKPOINT)
    else:
        raise FileNotFoundError(f"Checkpoint not found at: {PREV_RUN_CHECKPOINT}. "
                                "Please check the path or ensure the previous run's dataset is mounted.")
else:
    print("Starting training from scratch...")
    trainer.train()

## 5. Save PEFT Adapter Weights

In [ ]:
output_path = f"/kaggle/working/spurgeon_lora_epoch{RUN_NUMBER}"
print(f"Saving PEFT LoRA adapter weights to {output_path}...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print("Weights saved successfully.")